# LABORATORIO CALIFICADO N° 02
## Fundamentos de Gestión de Datos — Semana 4
**Docente:** Pilar Rocío Sayán Mejía | **Semestre:** 2026-I

---
## CASO: “Las Islas de Datos de HealthConnect”
**Protagonista:** Carlos Mendoza, DBA Junior
**Empresa:** HealthConnect — Clínica privada con 3 sedes en Lima

**Las 3 preguntas de la directora médica:**
1. ¿Cuántas citas tuvo cada médico este mes?
2. ¿Qué pacientes NO han tenido citas?
3. ¿Cuál es la especialidad con más diagnósticos?

---

## PASO 1: Generación de Datos Sintéticos de HealthConnect

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import random
from datetime import datetime, timedelta

np.random.seed(42)
random.seed(42)

# Tabla pacientes
nombres = ['Luis Perez','Maria Garcia','Carlos Mendoza','Sofia Quispe','Jorge Torres',
           'Ana Cruz','Pedro Saenz','Rosa Llanos','Miguel Huaman','Carmen Vega',
           'Diego Flores','Lucia Rios','Fernando Poma','Isabella Choque','Roberto Leon',
           'Valentina Soto','Andres Paredes','Daniela Mamani','Oscar Gutierrez','Silvia Ramos']
extra = [f'Paciente {i}' for i in range(21, 51)]
todos_nombres = nombres + extra

df_pacientes = pd.DataFrame({
    'id_paciente': [f'P{i:03d}' for i in range(1,51)],
    'nombre':      todos_nombres,
    'edad':        np.random.randint(5, 80, 50),
    'ciudad':      random.choices(['Lima','Arequipa','Cusco'], weights=[70,20,10], k=50),
    'telefono':    [f'9{random.randint(10000000,99999999)}' if random.random()>0.15 else None for _ in range(50)],
})

# Tabla medicos
especialidades = ['Cardiologia','Pediatria','Traumatologia','Medicina General','Dermatologia']
df_medicos = pd.DataFrame({
    'id_medico':     [f'M{i:02d}' for i in range(1,11)],
    'nombre':        [f'Dr(a). Medico {i}' for i in range(1,11)],
    'especialidad':  [especialidades[i%5] for i in range(10)],
    'anos_experiencia': np.random.randint(2, 25, 10),
})

# Tabla citas (40 pacientes con citas, 10 sin ninguna)
pacs_con_citas = [f'P{i:03d}' for i in range(1, 41)]
fechas_base = datetime(2026, 4, 1)
citas = []
for i in range(1, 81):
    citas.append({
        'id_cita':    f'CI{i:03d}',
        'id_paciente': random.choice(pacs_con_citas),
        'id_medico':   f'M{random.randint(1,10):02d}',
        'fecha':       (fechas_base + timedelta(days=random.randint(0,29))).strftime('%Y-%m-%d'),
        'estado':      random.choices(['Atendida','Pendiente','Cancelada'], weights=[65,25,10])[0],
    })
df_citas = pd.DataFrame(citas)

# Tabla diagnosticos (solo citas Atendidas)
citas_atendidas = df_citas[df_citas['estado']=='Atendida']['id_cita'].tolist()
codigos_cie = ['J06.9','I10','K29.7','M54.5','L30.9','J45.9','E11.9','N39.0']
descripciones = ['Infeccion respiratoria','Hipertension arterial','Gastritis','Lumbalgia',
                 'Dermatitis','Asma','Diabetes tipo 2','Infeccion urinaria']
diags = []
for idx, id_cita in enumerate(citas_atendidas[:60]):
    diags.append({
        'id_diagnostico': f'D{idx+1:03d}',
        'id_cita':        id_cita,
        'codigo_cie10':   random.choice(codigos_cie),
        'descripcion':    random.choice(descripciones),
    })
df_diagnosticos = pd.DataFrame(diags)

# Cargar en SQLite
conn = sqlite3.connect(':memory:')
conn.execute('PRAGMA foreign_keys = ON')
df_pacientes.to_sql('pacientes', conn, if_exists='replace', index=False)
df_medicos.to_sql('medicos', conn, if_exists='replace', index=False)
df_citas.to_sql('citas', conn, if_exists='replace', index=False)
df_diagnosticos.to_sql('diagnosticos', conn, if_exists='replace', index=False)

print('Datos cargados en SQLite:')
print(f'  Pacientes:    {len(df_pacientes)}')
print(f'  Medicos:      {len(df_medicos)}')
print(f'  Citas:        {len(df_citas)}')
print(f'  Diagnosticos: {len(df_diagnosticos)}')
display(df_citas.head())

## PASO 2: Diseño del Modelo ER

Complete la descripción de cada entidad y sus relaciones:

| Entidad | Atributo PK | Atributos principales | Relación con |
|---------|------------|----------------------|---------------|
| Paciente | id_paciente | nombre, edad, ciudad, telefono | Cita (1:N) |
| Medico | id_medico | nombre, especialidad | Cita (1:N) |
| Cita | id_cita | fecha, estado | Paciente (N:1), Medico (N:1), Diagnostico (1:0-1) |
| Diagnostico | id_diagnostico | codigo_cie10, descripcion | Cita (N:1) |

> **Tarea:** Dibuje el diagrama ER en papel o en draw.io y pegue la imagen en su entrega.
> Cardinalidades: Paciente 1 -- N Cita | Medico 1 -- N Cita | Cita 1 -- 0..1 Diagnostico

## PASO 3: DDL con Foreign Keys

In [ ]:
# DDL que se usaria para crear las tablas con integridad referencial
ddl = '''
CREATE TABLE IF NOT EXISTS pacientes (
    id_paciente TEXT PRIMARY KEY,
    nombre      TEXT NOT NULL,
    edad        INTEGER,
    ciudad      TEXT,
    telefono    TEXT
);

CREATE TABLE IF NOT EXISTS medicos (
    id_medico        TEXT PRIMARY KEY,
    nombre           TEXT NOT NULL,
    especialidad     TEXT,
    anos_experiencia INTEGER
);

CREATE TABLE IF NOT EXISTS citas (
    id_cita    TEXT PRIMARY KEY,
    id_paciente TEXT NOT NULL,
    id_medico   TEXT NOT NULL,
    fecha       TEXT,
    estado      TEXT,
    FOREIGN KEY (id_paciente) REFERENCES pacientes(id_paciente),
    FOREIGN KEY (id_medico)   REFERENCES medicos(id_medico)
);

CREATE TABLE IF NOT EXISTS diagnosticos (
    id_diagnostico TEXT PRIMARY KEY,
    id_cita        TEXT NOT NULL,
    codigo_cie10   TEXT,
    descripcion    TEXT,
    FOREIGN KEY (id_cita) REFERENCES citas(id_cita)
);
'''
print('DDL del sistema HealthConnect:')
print(ddl)

# Verificar estructura
for tabla in ['pacientes','medicos','citas','diagnosticos']:
    print(f'\nEstructura de {tabla}:')
    info = pd.read_sql_query(f'PRAGMA table_info({tabla})', conn)
    display(info[['name','type','notnull','pk']])

## PASO 4: INNER JOIN — Citas con Paciente y Médico

In [ ]:
q_inner = '''
SELECT c.id_cita,
       p.nombre  AS paciente,
       m.nombre  AS medico,
       m.especialidad,
       c.fecha,
       c.estado
FROM citas c
INNER JOIN pacientes p ON c.id_paciente = p.id_paciente
INNER JOIN medicos   m ON c.id_medico   = m.id_medico
ORDER BY c.fecha
LIMIT 15
'''
resultado_inner = pd.read_sql_query(q_inner, conn)
print(f'INNER JOIN: {len(resultado_inner)} registros (muestra de 15)')
display(resultado_inner)
print('\nNota: INNER JOIN excluye pacientes sin citas y citas sin medico registrado.')

## PASO 5: LEFT JOIN, GROUP BY + HAVING, CREATE VIEW

In [ ]:
# LEFT JOIN: pacientes SIN citas
q_left = '''
SELECT p.id_paciente, p.nombre, p.ciudad
FROM pacientes p
LEFT JOIN citas c ON p.id_paciente = c.id_paciente
WHERE c.id_cita IS NULL
'''
sin_citas = pd.read_sql_query(q_left, conn)
print(f'PACIENTES SIN CITAS: {len(sin_citas)}')
display(sin_citas)

# GROUP BY + HAVING: medicos con mas de 2 citas
q_grp = '''
SELECT m.nombre AS medico, m.especialidad,
       COUNT(c.id_cita) AS total_citas
FROM medicos m
LEFT JOIN citas c ON m.id_medico = c.id_medico
GROUP BY m.id_medico, m.nombre, m.especialidad
HAVING COUNT(c.id_cita) > 2
ORDER BY total_citas DESC
'''
medicos_activos = pd.read_sql_query(q_grp, conn)
print(f'\nMEDICOS CON MAS DE 2 CITAS:')
display(medicos_activos)

# CREATE VIEW resumen gerencial
conn.execute('DROP VIEW IF EXISTS vista_resumen_medico')
conn.execute('''
CREATE VIEW vista_resumen_medico AS
SELECT m.especialidad,
       COUNT(DISTINCT c.id_cita)       AS total_citas,
       COUNT(DISTINCT d.id_diagnostico) AS total_diagnosticos,
       COUNT(DISTINCT c.id_paciente)   AS pacientes_atendidos
FROM medicos m
LEFT JOIN citas c        ON m.id_medico = c.id_medico
LEFT JOIN diagnosticos d ON c.id_cita   = d.id_cita
GROUP BY m.especialidad
''')
print('\nVISTA RESUMEN GERENCIAL:')
display(pd.read_sql_query('SELECT * FROM vista_resumen_medico ORDER BY total_citas DESC', conn))

## PASO 6: Responder las 3 Preguntas de la Directora Médica

In [ ]:
# PREGUNTA 1: Cuantas citas tuvo cada medico este mes?
q1 = '''
SELECT m.nombre AS medico, m.especialidad,
       COUNT(c.id_cita) AS citas_del_mes
FROM medicos m
LEFT JOIN citas c ON m.id_medico = c.id_medico
    AND c.fecha BETWEEN '2026-04-01' AND '2026-04-30'
GROUP BY m.id_medico, m.nombre, m.especialidad
ORDER BY citas_del_mes DESC
'''
print('PREGUNTA 1: Citas por medico (abril 2026):')
display(pd.read_sql_query(q1, conn))

# PREGUNTA 2: Pacientes sin citas
q2 = '''
SELECT p.id_paciente, p.nombre, p.ciudad, p.telefono
FROM pacientes p
LEFT JOIN citas c ON p.id_paciente = c.id_paciente
WHERE c.id_cita IS NULL
ORDER BY p.nombre
'''
print('\nPREGUNTA 2: Pacientes sin ninguna cita registrada:')
display(pd.read_sql_query(q2, conn))

# PREGUNTA 3: Especialidad con mas diagnosticos
q3 = '''
SELECT m.especialidad,
       COUNT(d.id_diagnostico) AS total_diagnosticos
FROM medicos m
INNER JOIN citas c        ON m.id_medico = c.id_medico
INNER JOIN diagnosticos d ON c.id_cita   = d.id_cita
GROUP BY m.especialidad
ORDER BY total_diagnosticos DESC
'''
print('\nPREGUNTA 3: Especialidad con mas diagnosticos:')
display(pd.read_sql_query(q3, conn))

## ACTIVIDAD 3: Análisis Reflexivo

Responda con mínimo 3 líneas:

**A.** ¿Cuándo usarías INNER JOIN vs LEFT JOIN en HealthConnect? Dé un ejemplo de cada uno.

*(Su respuesta aqui)*

---

**B.** ¿Por qué es importante definir FK antes de cargar datos? ¿Qué problemas evita?

*(Su respuesta aqui)*

---

**C.** ¿Qué ventaja tiene usar una VIEW para los reportes de la directora médica?

*(Su respuesta aqui)*

---

**D.** Si agregaras la tabla 'Medicamentos', ¿qué relaciones tendría con las demás tablas?

*(Su respuesta aqui)*

## CONCLUSIONES

1. *(Conclusión 1)*

2. *(Conclusión 2)*

3. *(Conclusión 3)*

---
**Docente:** Pilar Rocío Sayán Mejía | **FGD 2026-I** | **LAB-C2**